# NIFTY Gap Strategy — v9 (v4.3 Combos + sim_cache, Apple-to-Apple)

**Purpose:** Reproduce v4.3's exact signal logic (top-10 bearish combos from v2_reliable_signals.csv)
using the same `sim_cache.csv` that powers v7 and v8, so all three versions are
evaluated on identical underlying trade outcomes. The only variable is **which days are filtered in**.

| Version | Signal filter | Training/OOS period | Sim source |
|---|---|---|---|
| v4.3 | Top-10 DOWN combos (in-sample) | Full 2024–2026 | Re-simulated |
| v7   | L1 logistic (continuous features) | OOS Jul 2025–Mar 2026 | sim_cache |
| v8   | Walk-forward L1/RF | OOS Jan 2025–Mar 2026 | sim_cache |
| **v9** | **Top-10 DOWN combos (same as v4.3)** | **Full 2024–2026** | **sim_cache** |

**Key caveat:** The 10 combos were selected on the same 2024–2026 data they are evaluated on.
v9 results are **in-sample** and serve as a ceiling estimate, not a forward-looking estimate.

| Parameter | Value |
|---|---|
| SL | −15% of entry premium |
| TP | +40% of entry premium |
| Strike | ATM − 50 (1-OTM PUT) |
| Exit | 11:15 AM hard stop |
| Breakeven win rate | 27.3% |
| Combos | Top 10 Direction=DOWN from v2_reliable_signals.csv |
| Sim source | v6/sim_cache.csv (instant load, no re-simulation) |

In [1]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import date

warnings.filterwarnings('ignore')

# ── Trade parameters (identical to v7 / v8) ───────────────────────────────────
SL_PCT           = 0.15
TP_PCT           = 0.40
LOT_SIZE         = 75
BASE_LOTS        = 5
MAX_LOTS         = 25
DTE0_MAX_LOTS    = 10
STARTING_CAPITAL = 200_000.0
BREAKEVEN        = SL_PCT / (SL_PCT + TP_PCT)

# ── Gate thresholds ────────────────────────────────────────────────────────────
VIX_PCT_MAX  = 0.65   # skip when VIX_INDIA rolling-252d percentile > 65th
GAP_NORM_MIN = 0.10   # skip if |gap / realized_vol| < 0.10 (no catalyst)
GAP_NORM_MAX = 0.80   # skip if |gap / realized_vol| > 0.80 (momentum runaway)

# ── Combo selection (replicating v4.3) ───────────────────────────────────────
TOP_N = 10   # top N bearish (Direction=DOWN) combos by edge_pp

# ── Comparison reference periods ─────────────────────────────────────────────
V7_OOS_START = date(2025, 7, 1)   # v7 OOS period start
V8_OOS_START = date(2025, 1, 1)   # v8 walk-forward OOS start

print('Config loaded.')
print(f'Breakeven win rate : {BREAKEVEN:.1%}')
print(f'Lot sizing         : BASE={BASE_LOTS}, MAX={MAX_LOTS}, DTE0_MAX={DTE0_MAX_LOTS}')
print(f'VIX gate           : VIX_INDIA_pct ≤ {VIX_PCT_MAX}')
print(f'Gap gate           : |gap_normalized| ∈ [{GAP_NORM_MIN}, {GAP_NORM_MAX}]')
print(f'Using top {TOP_N} DOWN combos from v2_reliable_signals.csv (v4.3 exact filter)')

Config loaded.
Breakeven win rate : 27.3%
Lot sizing         : BASE=5, MAX=25, DTE0_MAX=10
VIX gate           : VIX_INDIA_pct ≤ 0.65
Gap gate           : |gap_normalized| ∈ [0.1, 0.8]
Using top 10 DOWN combos from v2_reliable_signals.csv (v4.3 exact filter)


In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
GAP_TRADING    = Path.cwd().parent
SIM_CACHE_PATH = GAP_TRADING / 'v6' / 'sim_cache.csv'
RELIABLE_CSV   = GAP_TRADING / 'v2' / 'v2_reliable_signals.csv'

for lbl, p in [('sim_cache (v6)', SIM_CACHE_PATH), ('reliable_signals (v2)', RELIABLE_CSV)]:
    print(f'{lbl:<24}: {"OK" if p.exists() else "MISSING"}  ({p})')

# ── Load sim_cache ────────────────────────────────────────────────────────────
sim_df = pd.read_csv(SIM_CACHE_PATH, parse_dates=['date'])
sim_df['date'] = sim_df['date'].dt.date

# Parse boolean columns robustly (read_csv may load True/False as strings)
bool_map    = {'True': True, 'False': False, True: True, False: False}
signal_cols = [c for c in sim_df.columns
               if c not in ['date', 'win', 'exit_reason', 'entry_prem', 'exit_prem', 'dte']]
for col in signal_cols:
    sim_df[col] = sim_df[col].map(bool_map)
sim_df['win'] = sim_df['win'].map(bool_map)

print(f'\nsim_cache : {len(sim_df)} rows  ({sim_df["date"].min()} → {sim_df["date"].max()})')
print(f'Signals   : {signal_cols}')

# ── Load v4.3 combos from v2_reliable_signals.csv ────────────────────────────
reliable   = pd.read_csv(RELIABLE_CSV)
bear_rows  = reliable[reliable['Direction'] == 'DOWN'].head(TOP_N).reset_index(drop=True)
combos_raw = bear_rows['Signal'].tolist()
combos     = [[s.strip() for s in c.split(' + ')] for c in combos_raw]

print(f'\nTop {TOP_N} bearish combos (v4.3 filter):')
print(f'{"#":<4} {"N":>5} {"Edge_pp":>8}  Signal')
print('-' * 70)
for i, row in bear_rows.iterrows():
    print(f'{i+1:<4} {int(row["N"]):>5} {row["Edge_pp"]:>7.1f}pp  {row["Signal"]}')

# ── Apply combo filter ────────────────────────────────────────────────────────
def any_combo_fires(row):
    """Returns True if ALL signals in ANY of the top-10 combos are True."""
    for combo in combos:
        if all(row[sig] for sig in combo):
            return True
    return False

sim_df['fires'] = sim_df.apply(any_combo_fires, axis=1)
traded_df = sim_df[sim_df['fires']].reset_index(drop=True)

total_days    = len(sim_df)
fire_days     = len(traded_df)
base_win_all  = sim_df['win'].mean()
base_win_fire = traded_df['win'].mean()

print(f'\nCoverage      : {fire_days}/{total_days} days fire  ({fire_days/total_days:.1%} of tradeable days)')
print(f'Base win rate : {base_win_all:.1%}  (all {total_days} days)')
print(f'Fired win rate: {base_win_fire:.1%}  ({fire_days} days)  ← edge = {(base_win_fire-base_win_all)*100:+.1f}pp')
print(f'Breakeven     : {BREAKEVEN:.1%}')

sim_cache (v6)          : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v6\sim_cache.csv)
reliable_signals (v2)   : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v2\v2_reliable_signals.csv)

sim_cache : 402 rows  (2024-01-02 → 2026-03-24)
Signals   : ['Gap Up', 'Gap Up Strong', 'Gap Down', 'Prev India UP', 'Prev India DOWN', 'US UP', 'US DOWN', 'SGX UP', 'SGX DOWN', 'DAX UP', 'VIX Rising', 'VIX Falling', 'VIX Spike']

Top 10 bearish combos (v4.3 filter):
#        N  Edge_pp  Signal
----------------------------------------------------------------------
1       67    20.0pp  Gap Up + Prev India DOWN + US UP + SGX UP
2       54    19.4pp  Gap Up + Prev India DOWN + SGX UP + DAX UP
3       44    18.1pp  Gap Up + Prev India DOWN + SGX UP + VIX Falling
4       73    17.9pp  Gap Up + Prev India DOWN + SGX UP
5       58    16.0pp  Gap Up + Prev India DOWN + US UP + DAX UP
6      100    15.3pp  Gap Up 

In [3]:
# ── Gate features from aligned dataset ────────────────────────────────────────
ALIGNED_CSV = GAP_TRADING / 'v2' / 'v2_aligned_dataset.csv'
print(f'aligned : {"OK" if ALIGNED_CSV.exists() else "MISSING"}  ({ALIGNED_CSV})')

aligned = pd.read_csv(ALIGNED_CSV, parse_dates=['india_date'])
aligned = aligned.rename(columns={'india_date': 'date'})
aligned['date'] = aligned['date'].dt.date

nifty_vol                = aligned['prev_india_ret'].rolling(20).std()
aligned['VIX_INDIA_pct'] = aligned['VIX_INDIA_level'].rolling(252, min_periods=60).rank(pct=True)
aligned['gap_normalized'] = aligned['gap_pct'] / nifty_vol.replace(0, np.nan)

gate_cols = aligned[['date', 'VIX_INDIA_pct', 'gap_normalized']].copy()

# Merge gate features into sim_df and traded_df
sim_df    = sim_df.merge(gate_cols, on='date', how='left')
traded_df = traded_df.merge(gate_cols, on='date', how='left')

traded_df['gate_vix']  = traded_df['VIX_INDIA_pct'] <= VIX_PCT_MAX
traded_df['gate_gap']  = traded_df['gap_normalized'].abs().between(GAP_NORM_MIN, GAP_NORM_MAX)
traded_df['gate_both'] = traded_df['gate_vix'] & traded_df['gate_gap']

n = len(traded_df)
print(f'\nGate diagnostics on {n} combo-fired days:')
print(f'  VIX gate  (VIX_INDIA_pct ≤ {VIX_PCT_MAX})              : {int(traded_df["gate_vix"].sum()):>3} pass  ({traded_df["gate_vix"].mean():.0%})')
print(f'  Gap gate  (|gap_norm| ∈ [{GAP_NORM_MIN}, {GAP_NORM_MAX}])  : {int(traded_df["gate_gap"].sum()):>3} pass  ({traded_df["gate_gap"].mean():.0%})')
print(f'  Both gates combined                           : {int(traded_df["gate_both"].sum()):>3} pass  ({traded_df["gate_both"].mean():.0%})')

# Win rates per gate subset
for lbl, mask in [('All combo-fired', slice(None)),
                  ('VIX gate',        traded_df['gate_vix']),
                  ('Gap gate',        traded_df['gate_gap']),
                  ('Both gates',      traded_df['gate_both'])]:
    sub = traded_df[mask] if not isinstance(mask, slice) else traded_df
    wr  = sub['win'].mean() if len(sub) else float('nan')
    print(f'  {lbl:<22}: win rate {wr:.1%}  (N={len(sub)})')

aligned : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v2\v2_aligned_dataset.csv)

Gate diagnostics on 120 combo-fired days:
  VIX gate  (VIX_INDIA_pct ≤ 0.65)              :  56 pass  (47%)
  Gap gate  (|gap_norm| ∈ [0.1, 0.8])  :  92 pass  (77%)
  Both gates combined                           :  45 pass  (38%)
  All combo-fired       : win rate 31.7%  (N=120)
  VIX gate              : win rate 30.4%  (N=56)
  Gap gate              : win rate 30.4%  (N=92)
  Both gates            : win rate 28.9%  (N=45)


In [4]:
# ── Compounding backtest engine (identical to v7 / v8) ────────────────────────
def round_trip_charges(entry_prem: float, exit_prem: float, lots: int) -> float:
    buy_val  = entry_prem * lots * LOT_SIZE
    sell_val = exit_prem  * lots * LOT_SIZE
    brok  = 20.0 * 2
    stamp = 0.00003  * buy_val
    stt   = 0.000625 * sell_val
    exch  = 0.00053  * (buy_val + sell_val)
    sebi  = 0.000001 * (buy_val + sell_val)
    gst   = 0.18     * (brok + exch + sebi)
    return round(brok + stamp + stt + exch + sebi + gst, 2)


def run_backtest(df: pd.DataFrame, label: str, start_cap: float = STARTING_CAPITAL):
    """Capital-compounding backtest on a pre-filtered subset of sim_cache rows."""
    capital, peak = start_cap, start_cap
    ledger        = []

    for _, row in df.iterrows():
        ep  = float(row['entry_prem'])
        xp  = float(row['exit_prem'])
        dte = int(row['dte'])

        cost_lot = ep * LOT_SIZE
        if cost_lot <= 0:
            continue
        lots = max(BASE_LOTS, int(capital // cost_lot))
        lots = min(lots, MAX_LOTS)
        if dte == 0:
            lots = min(lots, DTE0_MAX_LOTS)

        charges   = round_trip_charges(ep, xp, lots)
        trade_pnl = (xp - ep) * LOT_SIZE * lots - charges
        capital  += trade_pnl
        peak      = max(peak, capital)
        dd_pct    = (peak - capital) / peak * 100

        ledger.append({
            'Trade#'   : len(ledger) + 1,
            'Date'     : row['date'],
            'DTE'      : dte,
            'Lots'     : lots,
            'Entry'    : ep,
            'Exit'     : xp,
            'PnL(pts)' : round(xp - ep, 2),
            'Charges'  : charges,
            'Trade PnL': round(trade_pnl, 2),
            'Capital'  : round(capital, 2),
            'DD%'      : round(dd_pct, 2),
            'Win'      : bool(row['win']),
            'Reason'   : row['exit_reason'],
        })

    if not ledger:
        print(f'{label}: 0 trades.')
        return pd.DataFrame(), start_cap

    led      = pd.DataFrame(ledger)
    wins     = int(led['Win'].sum())
    total    = len(led)
    roi      = (capital - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    max_dd   = led['DD%'].max()
    avg_win  = led.loc[led['Trade PnL'] > 0,  'Trade PnL'].mean() if wins > 0        else 0.0
    avg_loss = led.loc[led['Trade PnL'] <= 0, 'Trade PnL'].mean() if wins < total    else 0.0

    print(f'\n{"="*64}')
    print(f'  {label}')
    print(f'{"="*64}')
    print(f'  Trades       : {total}')
    print(f'  Win rate     : {wins/total*100:.1f}%  (breakeven: {BREAKEVEN:.1%}, base: {base_win_all:.1%})')
    print(f'  ROI          : {roi:+.1f}%')
    print(f'  Max drawdown : {max_dd:.1f}%')
    print(f'  End capital  : Rs {capital:,.0f}')
    print(f'  Avg win/loss : Rs {avg_win:,.0f} / Rs {avg_loss:,.0f}')
    print(led['Reason'].value_counts().to_string())
    print(f'{"="*64}')
    print(led[['Trade#', 'Date', 'DTE', 'Lots', 'Entry', 'Exit',
               'PnL(pts)', 'Trade PnL', 'Capital', 'DD%', 'Reason']].to_string(index=False))
    return led, capital


# ── Run full period ───────────────────────────────────────────────────────────
ledger_full, cap_full = run_backtest(
    traded_df,
    f'v9  Full period  {sim_df["date"].min()} → {sim_df["date"].max()}  (in-sample combos)'
)

# ── Year-by-year breakdown ────────────────────────────────────────────────────
if not ledger_full.empty:
    ledger_full['Year'] = pd.to_datetime(ledger_full['Date'].astype(str)).dt.year
    print(f'\nYear-by-year breakdown:')
    print(f'{"Year":<6} {"Trades":>7} {"Wins":>5} {"Win%":>6} {"PnL (Rs)":>12}')
    print('-' * 42)
    for yr in [2024, 2025, 2026]:
        sub = ledger_full[ledger_full['Year'] == yr]
        if sub.empty:
            print(f'{yr:<6} {"0":>7}')
            continue
        w  = int(sub['Win'].sum())
        n  = len(sub)
        pl = sub['Trade PnL'].sum()
        print(f'{yr:<6} {n:>7} {w:>5} {w/n*100:>5.1f}% {pl:>+12,.0f}')


  v9  Full period  2024-01-02 → 2026-03-24  (in-sample combos)
  Trades       : 120
  Win rate     : 31.7%  (breakeven: 27.3%, base: 24.9%)
  ROI          : +147.7%
  Max drawdown : 34.2%
  End capital  : Rs 495,448
  Avg win/loss : Rs 48,563 / Rs -20,589
Reason
Stop Loss     75
Target Hit    38
11:15 exit     7
 Trade#       Date  DTE  Lots  Entry   Exit  PnL(pts)  Trade PnL   Capital   DD%     Reason
      1 2024-01-05    6    25  82.10  94.30     12.20   22505.43 222505.43  0.00 11:15 exit
      2 2024-01-09    2    25  55.40  77.56     22.16   41252.59 263758.02  0.00 Target Hit
      3 2024-01-11    0    10  18.95  16.11     -2.84   -2201.65 261556.37  0.83  Stop Loss
      4 2024-01-12    6    25 105.05  89.29    -15.76  -29936.06 231620.31 12.18  Stop Loss
      5 2024-01-19    6    24 124.50 105.83    -18.67  -34038.76 197581.55 25.09  Stop Loss
      6 2024-01-23    2    25  61.70  86.38     24.68   45949.13 243530.68  7.67 Target Hit
      7 2024-02-02    6    25 128.70 109.

In [5]:
# ── OOS slice: same combos, restricted to Jul 2025 – Mar 2026 (v7 period) ────
# Note: this is NOT a genuine OOS test — combos were selected on full 2024-2026 data.
# Purpose: isolate signal filter effect from OOS period effect.

v7_oos_traded = traded_df[traded_df['date'] >= V7_OOS_START].reset_index(drop=True)
v8_oos_traded = traded_df[traded_df['date'] >= V8_OOS_START].reset_index(drop=True)

ledger_v7oos, _ = run_backtest(
    v7_oos_traded,
    'v9  Jul 2025–Mar 2026 only  (v7 period, combos still in-sample)'
)

ledger_v8oos, _ = run_backtest(
    v8_oos_traded,
    'v9  Jan 2025–Mar 2026 only  (v8 WF period, combos still in-sample)'
)


  v9  Jul 2025–Mar 2026 only  (v7 period, combos still in-sample)
  Trades       : 34
  Win rate     : 26.5%  (breakeven: 27.3%, base: 24.9%)
  ROI          : -57.9%
  Max drawdown : 77.3%
  End capital  : Rs 84,101
  Avg win/loss : Rs 19,745 / Rs -11,744
Reason
Stop Loss     22
Target Hit     9
11:15 exit     3
 Trade#       Date  DTE  Lots  Entry   Exit  PnL(pts)  Trade PnL   Capital   DD%     Reason
      1 2025-07-03    0    10  31.00  43.40     12.40    9196.80 209196.80  0.00 Target Hit
      2 2025-07-22    2    25  61.10  85.54     24.44   45501.84 254698.64  0.00 Target Hit
      3 2025-07-23    1    25  32.90  27.96     -4.94   -9415.82 245282.82  3.70  Stop Loss
      4 2025-07-30    1    25  58.15  49.43     -8.72  -16584.79 228698.03 10.21  Stop Loss
      5 2025-08-13    1    25  47.05  39.99     -7.06  -13436.47 215261.56 15.48  Stop Loss
      6 2025-09-04    5    25  68.85  58.52    -10.33  -19638.04 195623.52 23.19  Stop Loss
      7 2025-09-05    4    25  58.50  49.

In [6]:
# ── Gated backtests: VIX gate, gap gate, both gates ───────────────────────────
traded_vix  = traded_df[traded_df['gate_vix']].reset_index(drop=True)
traded_gap  = traded_df[traded_df['gate_gap']].reset_index(drop=True)
traded_both = traded_df[traded_df['gate_both']].reset_index(drop=True)

ledger_vix,  _ = run_backtest(traded_vix,
    f'v9 + VIX gate  (VIX_INDIA_pct ≤ {VIX_PCT_MAX})  full period  (in-sample combos)')
ledger_gap,  _ = run_backtest(traded_gap,
    f'v9 + Gap gate  (|gap_norm| ∈ [{GAP_NORM_MIN},{GAP_NORM_MAX}])  full period  (in-sample combos)')
ledger_both, _ = run_backtest(traded_both,
    f'v9 + Both gates  full period  (in-sample combos)')

# ── Gated OOS slices ──────────────────────────────────────────────────────────
def oos_slice(df, start):
    return df[df['date'] >= start].reset_index(drop=True)

ledger_vix_v7oos,   _ = run_backtest(oos_slice(traded_vix,  V7_OOS_START),
    f'v9 + VIX gate  Jul 2025–Mar 2026')
ledger_both_v7oos,  _ = run_backtest(oos_slice(traded_both, V7_OOS_START),
    f'v9 + Both gates  Jul 2025–Mar 2026')
ledger_both_v8oos,  _ = run_backtest(oos_slice(traded_both, V8_OOS_START),
    f'v9 + Both gates  Jan 2025–Mar 2026')


  v9 + VIX gate  (VIX_INDIA_pct ≤ 0.65)  full period  (in-sample combos)
  Trades       : 56
  Win rate     : 30.4%  (breakeven: 27.3%, base: 24.9%)
  ROI          : +44.3%
  Max drawdown : 56.8%
  End capital  : Rs 288,672
  Avg win/loss : Rs 40,878 / Rs -15,545
Reason
Stop Loss     36
Target Hit    17
11:15 exit     3
 Trade#       Date  DTE  Lots  Entry   Exit  PnL(pts)  Trade PnL   Capital   DD%     Reason
      1 2024-04-04    0    10  43.10  60.34     17.24   12804.94 212804.94  0.00 Target Hit
      2 2024-04-24    1    25  50.20  42.67     -7.53  -14327.88 198477.06  6.73  Stop Loss
      3 2024-04-26    6    24 106.30 148.82     42.52   76027.90 274504.96  0.00 Target Hit
      4 2024-04-30    2    25  74.75  63.54    -11.21  -21307.08 253197.88  7.76  Stop Loss
      5 2024-06-19    1    25  56.55  79.17     22.62   42109.89 295307.77  0.00 Target Hit
      6 2024-07-03    1    25  69.40  58.99    -10.41  -19789.82 275517.95  6.70  Stop Loss
      7 2024-07-04    0    10  38

In [7]:
# ── DTE / Expiry-change analysis ───────────────────────────────────────────────
# NIFTY weekly expiry moved Thursday → Tuesday on 2025-09-02.
# DTE distribution before: 0,1,2,6  |  after: 0,4,5,6
# This cell quantifies the structural break and isolates DTE=0 (Tuesday) trades.

EXPIRY_CHANGE = date(2025, 9, 2)

traded_df['regime']  = traded_df['date'].apply(
    lambda d: 'Tuesday expiry (new)' if d >= EXPIRY_CHANGE else 'Thursday expiry (old)')
traded_df['weekday'] = pd.to_datetime(traded_df['date'].astype(str)).dt.day_name()

# ── Win rate by DTE, split by regime ──────────────────────────────────────────
print('=== WIN RATE BY DTE  (before vs after expiry change) ===')
for regime in ['Thursday expiry (old)', 'Tuesday expiry (new)']:
    sub = traded_df[traded_df['regime'] == regime]
    tbl = sub.groupby('dte')['win'].agg(['mean', 'count']).rename(
          columns={'mean': 'WinRate', 'count': 'N'})
    print(f'\n  {regime}  (N={len(sub)}):')
    print(f'  {"DTE":>4}  {"N":>4}  {"WinRate":>8}')
    for dte, row in tbl.iterrows():
        flag = '  ← best' if row['WinRate'] == tbl['WinRate'].max() else ''
        print(f'  {dte:>4}  {int(row["N"]):>4}  {row["WinRate"]:>8.1%}{flag}')

# ── Weekday win rates after expiry change ─────────────────────────────────────
post = traded_df[traded_df['regime'] == 'Tuesday expiry (new)']
print(f'\n=== WEEKDAY WIN RATES AFTER SEP 2 2025  (N={len(post)}) ===')
wd_tbl = post.groupby('weekday')['win'].agg(['mean', 'count']).rename(
         columns={'mean': 'WinRate', 'count': 'N'})
day_order = ['Tuesday', 'Wednesday', 'Thursday', 'Friday']
for day in day_order:
    if day in wd_tbl.index:
        r = wd_tbl.loc[day]
        dte_note = {'Tuesday': 'DTE=0', 'Wednesday': 'DTE=6',
                    'Thursday': 'DTE=5', 'Friday': 'DTE=4'}.get(day, '')
        print(f'  {day:<10} ({dte_note})  N={int(r["N"]):>2}  WinRate={r["WinRate"]:.1%}')

# ── Combo win rate before vs after ────────────────────────────────────────────
pre  = traded_df[traded_df['regime'] == 'Thursday expiry (old)']
print(f'\nCombo win rate BEFORE Sep 2 2025 : {pre["win"].mean():.1%}  (N={len(pre)})')
print(f'Combo win rate AFTER  Sep 2 2025 : {post["win"].mean():.1%}  (N={len(post)})')
print(f'Breakeven                         : {BREAKEVEN:.1%}')

# ── Backtest: DTE=0 only, post-expiry-change ──────────────────────────────────
traded_dte0_new = traded_df[
    (traded_df['regime'] == 'Tuesday expiry (new)') & (traded_df['dte'] == 0)
].reset_index(drop=True)

traded_non_dte0 = traded_df[
    (traded_df['regime'] == 'Tuesday expiry (new)') & (traded_df['dte'] != 0)
].reset_index(drop=True)

print(f'\n=== BACKTEST: DTE=0 (Tuesday) only, post Sep 2 2025 ===')
ledger_dte0, _ = run_backtest(
    traded_dte0_new, 'v9 + DTE=0 filter  post Sep-2025  (IS combos)',
    start_cap=STARTING_CAPITAL)

print(f'\n=== BACKTEST: non-DTE=0, post Sep 2 2025 (what to SKIP) ===')
ledger_non_dte0, _ = run_backtest(
    traded_non_dte0, 'v9 non-DTE=0  post Sep-2025  (IS combos)',
    start_cap=STARTING_CAPITAL)

=== WIN RATE BY DTE  (before vs after expiry change) ===

  Thursday expiry (old)  (N=91):
   DTE     N   WinRate
     0    25     28.0%
     1    19     31.6%
     2    29     51.7%  ← best
     6    18     16.7%

  Tuesday expiry (new)  (N=29):
   DTE     N   WinRate
     0    11     45.5%  ← best
     4     6      0.0%
     5     7     14.3%
     6     5     20.0%

=== WEEKDAY WIN RATES AFTER SEP 2 2025  (N=29) ===
  Tuesday    (DTE=0)  N=11  WinRate=45.5%
  Wednesday  (DTE=6)  N= 5  WinRate=20.0%
  Thursday   (DTE=5)  N= 7  WinRate=14.3%
  Friday     (DTE=4)  N= 6  WinRate=0.0%

Combo win rate BEFORE Sep 2 2025 : 34.1%  (N=91)
Combo win rate AFTER  Sep 2 2025 : 24.1%  (N=29)
Breakeven                         : 27.3%

=== BACKTEST: DTE=0 (Tuesday) only, post Sep 2 2025 ===

  v9 + DTE=0 filter  post Sep-2025  (IS combos)
  Trades       : 11
  Win rate     : 45.5%  (breakeven: 27.3%, base: 24.9%)
  ROI          : +28.5%
  Max drawdown : 5.9%
  End capital  : Rs 256,956
  Avg win/loss

In [8]:
# ── Apple-to-apple comparison table ──────────────────────────────────────────
def _row(led, label, period):
    if led is None or led.empty:
        return f'  {label:<44} {period:<22} {"0":>7} {"N/A":>6} {"N/A":>8} {"N/A":>7}'
    n   = len(led)
    w   = int(led['Win'].sum())
    roi = (led['Capital'].iloc[-1] - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    dd  = led['DD%'].max()
    return f'  {label:<44} {period:<22} {n:>7} {w/n*100:>5.1f}% {roi:>+7.1f}% {dd:>6.1f}%'

SEP = '=' * 94

print()
print(SEP)
print('  APPLE-TO-APPLE COMPARISON  (all versions use identical sim_cache outcomes)')
print(SEP)
print(f'  {"Version":<44} {"Period":<22} {"Trades":>7} {"Win%":>6} {"ROI":>8} {"MaxDD":>7}')
print(f'  {"-"*44} {"-"*22} {"-"*7} {"-"*6} {"-"*8} {"-"*7}')

# ── Baseline: v9 without any gate ─────────────────────────────────────────────
print(_row(ledger_full,          'v9  v4.3 combos  (IN-SAMPLE)',           '2024-01 – 2026-03'))
print(_row(ledger_v7oos,         'v9  same combos, v7 period  (IS)',        '2025-07 – 2026-03'))
print(_row(ledger_v8oos,         'v9  same combos, v8 period  (IS)',        '2025-01 – 2026-03'))

# ── VIX / gap gates ───────────────────────────────────────────────────────────
print(f'  {"-"*44} {"-"*22} {"-"*7} {"-"*6} {"-"*8} {"-"*7}')
print(_row(ledger_vix,           'v9 + VIX gate  (IS)',                     '2024-01 – 2026-03'))
print(_row(ledger_gap,           'v9 + Gap gate  (IS)',                     '2024-01 – 2026-03'))
print(_row(ledger_both,          'v9 + Both gates  (IS)',                   '2024-01 – 2026-03'))
print(_row(ledger_both_v7oos,    'v9 + Both gates  v7 period  (IS)',        '2025-07 – 2026-03'))

# ── Expiry / DTE filter ───────────────────────────────────────────────────────
print(f'  {"-"*44} {"-"*22} {"-"*7} {"-"*6} {"-"*8} {"-"*7}')
print(_row(ledger_dte0,          'v9 + DTE=0 (Tue) post-expiry change (IS)','2025-09 – 2026-03'))
print(_row(ledger_non_dte0,      'v9   DTE!=0 post-expiry change  (IS)',    '2025-09 – 2026-03'))

# ── Genuine OOS references ────────────────────────────────────────────────────
print(f'  {"-"*44} {"-"*22} {"-"*7} {"-"*6} {"-"*8} {"-"*7}')
print(f'  {"v7  L1 logistic  (GENUINE OOS)":<44} {"2025-07 – 2026-03":<22} {10:>7} {"40.0%":>6} {"  +8.4%":>8} {" 21.0%":>7}')
print(f'  {"v8  L1 walk-forward  (GENUINE OOS)":<44} {"2025-01 – 2026-03":<22} {24:>7} {"25.0%":>6} {" -42.3%":>8} {" 47.1%":>7}')
print(f'  {"v8  RF  walk-forward  (GENUINE OOS)":<44} {"2025-01 – 2026-03":<22} { 8:>7} {"25.0%":>6} {"  -6.7%":>8} {" 22.0%":>7}')
print(f'  {"v11 Both gates  (GENUINE OOS)":<44} {"2025-07 – 2026-02":<22} {17:>7} {"47.1%":>6} {" +88.0%":>8} {" 15.4%":>7}')

print(SEP)
print()
print(f'  Breakeven win rate  : {BREAKEVEN:.1%}')
print(f'  Base win rate       : {base_win_all:.1%}  (all {total_days} tradeable days in sim_cache)')
print(f'  Combo win rate      : {base_win_fire:.1%}  ({fire_days} fired days)  — IN-SAMPLE edge')
print(f'  EXPIRY_CHANGE       : 2025-09-02  (Thursday → Tuesday weekly expiry)')
print()
print('  KEY FINDING: After the expiry change, the combo filter fires on DTE=4 (Fri)')
print('  and DTE=5 (Thu) which have 0% and 14% win rates under Tuesday expiry.')
print('  DTE=0 (Tuesday expiry day) has 45%+ win rate — only worthwhile day to trade.')
print()
print('  NOTE: all v9 rows are IN-SAMPLE (combos selected on 2024-2026 data).')
print('  Use v7/v8/v11 rows for unbiased OOS estimates.')


  APPLE-TO-APPLE COMPARISON  (all versions use identical sim_cache outcomes)
  Version                                      Period                  Trades   Win%      ROI   MaxDD
  -------------------------------------------- ---------------------- ------- ------ -------- -------
  v9  v4.3 combos  (IN-SAMPLE)                 2024-01 – 2026-03          120  31.7%  +147.7%   34.2%
  v9  same combos, v7 period  (IS)             2025-07 – 2026-03           34  26.5%   -57.9%   77.3%
  v9  same combos, v8 period  (IS)             2025-01 – 2026-03           59  28.8%   -12.8%   66.5%
  -------------------------------------------- ---------------------- ------- ------ -------- -------
  v9 + VIX gate  (IS)                          2024-01 – 2026-03           56  30.4%   +44.3%   56.8%
  v9 + Gap gate  (IS)                          2024-01 – 2026-03           92  30.4%  +119.1%   36.0%
  v9 + Both gates  (IS)                        2024-01 – 2026-03           45  28.9%   +20.5%   54.8%
  v9